# MechRabot — Retrieval Evaluation

In [ ]:
# Step 0: Install dependencies
!pip install FlagEmbedding qdrant-client ranx openpyxl --quiet
print("✅ Done")

In [ ]:
# Step 1: Connect to Qdrant
from qdrant_client import QdrantClient, models

QDRANT_URL = "YOUR_QDRANT_URL"
QDRANT_KEY = "YOUR_API_KEY"
COL_NAME   = "mechrabot_Vdb_1"

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_KEY)
print("✅ Connected")

In [ ]:
# Step 2: Load evaluation dataset
import json

FILE_PATH = "/kaggle/input/datasets/ahmedezzattaha/evaluations-query-v1/evaluation_30_V1.json"

with open(FILE_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# queries  : { qid -> query_text }
# qrels    : { qid -> {chunk_id: 1} }  <- ground truth
queries = {}
qrels   = {}

for q in eval_data["queries"]:
    qid         = q["id"]
    queries[qid] = q["query"]
    qrels[qid]  = {cid: 1 for cid in q["relevant_chunk_ids"]}

print(f"✅ Loaded {len(queries)} queries")

In [ ]:
# Step 3: Embed queries with BGE-M3
from FlagEmbedding import BGEM3FlagModel

qid_list   = sorted(queries.keys())
query_list = [queries[qid] for qid in qid_list]

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

embeddings = model.encode(
    query_list,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
    batch_size=64,
    max_length=512,
)
print(f"✅ Embedded {len(query_list)} queries")

In [ ]:
# Step 4: Hybrid search in Qdrant
K = 10
run_scores = {}  # { qid -> {chunk_id: score} }

for i, qid in enumerate(qid_list):
    hits = client.query_points(
        collection_name=COL_NAME,
        prefetch=[
            models.Prefetch(
                query=embeddings["dense_vecs"][i].tolist(),
                using="dense",
                limit=K * 2,
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=[int(k) for k in embeddings["lexical_weights"][i].keys()],
                    values=[float(v) for v in embeddings["lexical_weights"][i].values()],
                ),
                using="sparse",
                limit=K * 2,
            ),
        ],
        query=embeddings["dense_vecs"][i].tolist(),
        using="dense",
        limit=K,
    )
    run_scores[qid] = {str(p.id): p.score for p in hits.points}

print(f"✅ Searched {len(run_scores)} queries")

In [ ]:
# Step 5: Compute metrics with ranx
from ranx import Qrels, Run, evaluate

qrels_str = {qid: {str(k): v for k, v in rels.items()} for qid, rels in qrels.items()}

results = evaluate(
    Qrels(qrels_str),
    Run(run_scores),
    ["mrr@10", "ndcg@10", "recall@10", "hit_rate@10", "precision@10"]
)

for metric, score in results.items():
    print(f"  {metric:<18} = {score:.4f}")

In [ ]:
# Step 6: Save results to JSON
import json
from datetime import datetime

output = {
    "timestamp" : datetime.now().strftime("%Y%m%d_%H%M"),
    "model"     : "BAAI/bge-m3",
    "collection": COL_NAME,
    "k"         : K,
    "overall"   : {k: round(v, 4) for k, v in results.items()},
}

out_path = f"/kaggle/working/eval_results_{output['timestamp']}.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✅ Saved → {out_path}")